# ED Patient Flow Prediction (Machine Learning)

This notebook builds a machine learning model to predict high-acuity emergency department patients using clinical and demographic data.

- Dataset: Hospital Triage and Patient History (Kaggle)
- Target: High Acuity (ESI ≤ 2)

In [2]:
import numpy as np
import pandas as pd
import random
import os

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = "42"

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import joblib

In [4]:
if os.path.exists("../data/ed_patient_flow.csv"):
    df = pd.read_csv("../data/ed_patient_flow.csv")
    print("Loaded FULL dataset")
else:
    df = pd.read_csv("../data/sample_ed_data.csv")
    print("Loaded SAMPLE dataset")

print("Dataset shape:", df.shape)

Loaded FULL dataset
Dataset shape: (32946, 972)


In [5]:
df["esi_num"] = pd.to_numeric(df["esi"], errors="coerce")
df = df.dropna(subset=["esi_num"])

df["high_acuity"] = (df["esi_num"] <= 2).astype(int)

print(df["high_acuity"].value_counts(normalize=True))

high_acuity
0    0.698626
1    0.301374
Name: proportion, dtype: float64


In [6]:
df = df.drop(columns=["esi", "esi_num"])

In [7]:
X = df.drop(columns=["high_acuity"])
y = df["high_acuity"]

X = X.copy()
X["age"] = pd.to_numeric(X["age"], errors="coerce")

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

In [9]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [10]:
log_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=SEED
    ))
])

log_reg.fit(X_train, y_train)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['benzodiazepinesscreen,urine,noconf._last' 'epithelialcells_last'
 'phencyclidine(pcp)screen,urine,noconf._last'
 'benzodiazepinesscreen,urine,noconf._min' 'epithelialcells_min'
 'phencyclidine(pcp)screen,urine,noconf._min'
 'benzodiazepinesscreen,urine,noconf._max' 'epithelialcells_max'
 'phencyclidine(pcp)screen,urine,noconf._max'
 'benzodiazepinesscreen,urine,noconf._median' 'epithelialcells_median'
 'phencyclidine(pcp)screen,urine,noconf._median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/prepro

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', '2ndarymalig',
                                                   'abdomhernia', 'abdomnlpain',
                                                   'abortcompl', 'acqfootdef',
                                                   'acrenlfail', 'acutecvd',
                                                   'acutemi', 'acutphanm',
                                                   'adjustmentdisorders',
                                                   'adltrespfl',
                                                   'alcoholrelateddisorders',
                                                   'allergy', 'amniosdx',
                                                   'analrectal',...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['dep_name', 'gender',
                                                   'ethnicity', 'race', 'lang',
                                                   'religion', 'maritalstatus',
                                                   'employstatus',
                                                   'insurance_status',
                                                   'disposition', 'arrivalmode',
                                                   'arrivalmonth', 'arrivalday',
                                                   'arrivalhour_bin',
                                                   'previousdispo'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [11]:
y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]

print("Logistic Regression Results:\n")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['benzodiazepinesscreen,urine,noconf._last' 'epithelialcells_last'
 'phencyclidine(pcp)screen,urine,noconf._last'
 'benzodiazepinesscreen,urine,noconf._min' 'epithelialcells_min'
 'phencyclidine(pcp)screen,urine,noconf._min'
 'benzodiazepinesscreen,urine,noconf._max' 'epithelialcells_max'
 'phencyclidine(pcp)screen,urine,noconf._max'
 'benzodiazepinesscreen,urine,noconf._median' 'epithelialcells_median'
 'phencyclidine(pcp)screen,urine,noconf._median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['benzodiazepinesscreen,urine,noconf._last' 'epithelialcells_last'
 'phencyclidine(pcp)screen,urine,noconf._last'
 'benzodiazepinesscreen,urine,noconf._min' 'epithelialcells_min'


Logistic Regression Results:

[[3584 1002]
 [ 607 1372]]
              precision    recall  f1-score   support

           0       0.86      0.78      0.82      4586
           1       0.58      0.69      0.63      1979

    accuracy                           0.75      6565
   macro avg       0.72      0.74      0.72      6565
weighted avg       0.77      0.75      0.76      6565

ROC-AUC: 0.8038686628262257


In [12]:
threshold = 0.40
y_pred_adj = (y_proba >= threshold).astype(int)

print("Threshold = 0.40 Results:\n")
print(confusion_matrix(y_test, y_pred_adj))
print(classification_report(y_test, y_pred_adj))

Threshold = 0.40 Results:

[[3184 1402]
 [ 426 1553]]
              precision    recall  f1-score   support

           0       0.88      0.69      0.78      4586
           1       0.53      0.78      0.63      1979

    accuracy                           0.72      6565
   macro avg       0.70      0.74      0.70      6565
weighted avg       0.77      0.72      0.73      6565



In [13]:
rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['benzodiazepinesscreen,urine,noconf._last' 'epithelialcells_last'
 'phencyclidine(pcp)screen,urine,noconf._last'
 'benzodiazepinesscreen,urine,noconf._min' 'epithelialcells_min'
 'phencyclidine(pcp)screen,urine,noconf._min'
 'benzodiazepinesscreen,urine,noconf._max' 'epithelialcells_max'
 'phencyclidine(pcp)screen,urine,noconf._max'
 'benzodiazepinesscreen,urine,noconf._median' 'epithelialcells_median'
 'phencyclidine(pcp)screen,urine,noconf._median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', '2ndarymalig',
                                                   'abdomhernia', 'abdomnlpain',
                                                   'abortcompl', 'acqfootdef',
                                                   'acrenlfail', 'acutecvd',
                                                   'acutemi', 'acutphanm',
                                                   'adjustmentdisorders',
                                                   'adltrespfl',
                                                   'alcoholrelateddisorders',
                                                   'allergy', 'amniosdx',
                                                   'analrectal',...
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['dep_name', 'gender',
                                                   'ethnicity', 'race', 'lang',
                                                   'religion', 'maritalstatus',
                                                   'employstatus',
                                                   'insurance_status',
                                                   'disposition', 'arrivalmode',
                                                   'arrivalmonth', 'arrivalday',
                                                   'arrivalhour_bin',
                                                   'previousdispo'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        min_samples_leaf=50, n_estimators=200,
                                        n_jobs=-1, random_state=42))])

In [14]:
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Results:\n")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['benzodiazepinesscreen,urine,noconf._last' 'epithelialcells_last'
 'phencyclidine(pcp)screen,urine,noconf._last'
 'benzodiazepinesscreen,urine,noconf._min' 'epithelialcells_min'
 'phencyclidine(pcp)screen,urine,noconf._min'
 'benzodiazepinesscreen,urine,noconf._max' 'epithelialcells_max'
 'phencyclidine(pcp)screen,urine,noconf._max'
 'benzodiazepinesscreen,urine,noconf._median' 'epithelialcells_median'
 'phencyclidine(pcp)screen,urine,noconf._median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['benzodiazepinesscreen,urine,noconf._last' 'epithelialcells_last'
 'phencyclidine(pcp)screen,urine,noconf._last'
 'benzodiazepinesscreen,urine,noconf._min' 'epithelialcells_min'


Random Forest Results:

[[3494 1092]
 [ 460 1519]]
              precision    recall  f1-score   support

           0       0.88      0.76      0.82      4586
           1       0.58      0.77      0.66      1979

    accuracy                           0.76      6565
   macro avg       0.73      0.76      0.74      6565
weighted avg       0.79      0.76      0.77      6565

ROC-AUC: 0.8351316163810723


In [15]:
model = rf.named_steps["model"]
feature_names = rf.named_steps["preprocessor"].get_feature_names_out()

importances = pd.Series(model.feature_importances_, index=feature_names)
importances.sort_values(ascending=False).head(15)

cat__disposition_Admit        0.108616
cat__disposition_Discharge    0.098881
cat__arrivalmode_ambulance    0.065124
cat__dep_name_A               0.061155
cat__dep_name_C               0.045728
cat__arrivalmode_Car          0.032104
num__age                      0.025046
num__meds_cardiovascular      0.017050
cat__employstatus_Retired     0.012936
num__n_admissions             0.011134
cat__dep_name_B               0.010800
num__spo2_min                 0.010749
num__triage_vital_hr          0.010703
num__cc_chestpain             0.010441
num__ekg_count                0.009732
dtype: float64

In [16]:
import os
import joblib

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "rf_pipeline.pkl")

joblib.dump(rf, model_path)

print(f"Model saved at: {model_path}")

Model saved at: ../models/rf_pipeline.pkl
